In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
import pickle

path = os.getcwd().split(os.sep + 'GUI')[0]
if path not in sys.path:
    print("not here")
    sys.path.append(path)

from neurolib.models.aln import ALNModel
from neurolib.utils import plotFunctions as plotFunc
from neurolib.utils import costFunctions as cost
import neurolib.dashboard.functions as functions
import neurolib.dashboard.data as data
    
# This will reload all imports as soon as the code changes
%load_ext autoreload
%autoreload 2 

#path = os.path.join(os.getcwd(), "plots")

not here


In [2]:
# read case
print(os.getcwd())
case = os.getcwd().split(os.sep)[-1]
print(case[0])

/mnt/antares_raid/home/salfenmoser/neurolib/GUI/current/gui/data/00011
0


### Bistability

In [3]:
aln = ALNModel()
N = aln.params.N

data.set_parameters(aln)

state_vars = aln.state_vars
init_vars = aln.init_vars

##############################################################
def setinit(init_vars_, model):
    state_vars = model.state_vars
    init_vars = model.init_vars
    for iv in range(len(init_vars)):
        for sv in range(len(state_vars)):
            if state_vars[sv] in init_vars[iv]:
                #print("set init vars ", )
                if model.params[init_vars[iv]].ndim == 2:
                    model.params[init_vars[iv]][0,:] = init_vars_[sv]
                else:
                    model.params[init_vars[iv]][0] = init_vars_[sv]
                    
##############################################################               
def setmaxmincontrol(max_c_c, min_c_c, max_c_r, min_c_r):
    import numpy as np
    
    max_cntrl = np.zeros(( 6 ))
    min_cntrl = np.zeros(( 6 ))
    
    max_cntrl[0] = max_c_c
    min_cntrl[0] = min_c_c
    max_cntrl[1] = max_c_c
    min_cntrl[1] = min_c_c
    max_cntrl[2] = max_c_r
    min_cntrl[2] = min_c_r
    max_cntrl[3] = max_c_r
    min_cntrl[3] = min_c_r
    max_cntrl[4] = max_c_r
    min_cntrl[4] = min_c_r
    max_cntrl[5] = max_c_r
    min_cntrl[5] = min_c_r
            
    return max_cntrl, min_cntrl

#####################################################
def getclosest(k_, found_solution, exc, inh, already_tried_):
    import numpy as np
    if len(found_solution) == 0:
        print("no solutions found")
        return -1
    
    start_ind = -1
    for j_ in found_solution:
        if j_ not in already_tried_ and j_ != k_:
            start_ind = j_
            break
            
    if start_ind == -1:
        return -1
        
    min_dist = np.sqrt((exc[k_] - exc[start_ind])**2 + (inh[k_] - inh[start_ind])**2)
    min_i = start_ind
        
    print(found_solution, already_tried_)
        
    if len(found_solution) == len(already_tried_):
        print("already tried all options")
        min_i = -1
        return min_i
    
    for i_ in found_solution:
        if i_ not in already_tried_:
            if i_ != k_ and i_ != min_i:
                dist_ = np.sqrt((exc[k_] - exc[i_])**2 + (inh[k_] - inh[i_])**2)
                if dist_ < min_dist:
                    min_dist = dist_
                    min_i = i_
                    
    if min_i == 0 and 0 in already_tried_:
        return -1
    
    return min_i

In [4]:
##### LOAD BOUNDARIES
data_file = 'bi.pickle'
with open(data_file,'rb') as f:
    load_array= pickle.load(f)
exc = load_array[0]
inh = load_array[1]
print(len(exc))
#plt.scatter(exc, inh)

147


In [5]:
bestControl_init = [None] * len(exc)
bestState_init = [None] * len(exc)
cost_init = [None] * len(exc)
runtime_init = [None] * len(exc)
grad_init = [None] * len(exc)
phi_init = [None] * len(exc)
costnode_init = [None] * len(exc)
weights_init = [None] * len(exc)

conv_init = [[False]*2] * len(exc)

In [6]:
bestControl_0 = [None] * len(exc)
bestState_0 = [None] * len(exc)
cost_0 = [None] * len(exc)
runtime_0 = [None] * len(exc)
grad_0 = [None] * len(exc)
phi_0 = [None] * len(exc)
costnode_0 = [None] * len(exc)
weights_0 = [None] * len(exc)

conv_0 = [[False]*2] * len(exc)

cgv_list = [None, "HS", "FR", "PR", "CD", "LS", "DY", "WYL", "HZ", None]

In [7]:
initVars = [None] * len(exc)
target = [None] * len(exc)
cost_uncontrolled = [None] * len(exc)

In [8]:
dur_pre = 10
dur_post = 10

n_pre = int(np.around(dur_pre/aln.params.dt + 1.,1))
n_post = int(np.around(dur_post/aln.params.dt + 1.,1))

tol = 1e-32
start_step = 10.
c_scheme = np.zeros(( 1,1 ))
c_scheme[0,0] = 1.
u_mat = np.identity(1)
u_scheme = np.array([[1.]])

c_var = [ [0], [1], [0,1]]
p_var = [ [0], [0], [0]]

### CURRENTS
cntrl_vars_0 = [0,1]
prec_vars = [0]

if case[0] == '0':    # low to high
    max_I = [3., -3.]
elif case[0] == '1':
    max_I = [-3., 3.]
    
if case[1] == '0':    # sparsity
    factor_ws = 1.
    factor_we = 0.
elif case[1] == '1':  # energy
    factor_ws = 0.
    factor_we = 1.
    
if case[3] == '0' and case[2] == '0':
    cntrl_vars_init = [0]
elif case[3] == '1' and case[2] == '0':
    cntrl_vars_init = [1]
elif case[3] == '0' and case[2] == '1':
    cntrl_vars_init = [2,4]
elif case[3] == '1' and case[2] == '1':
    cntrl_vars_init = [3,5]

if case[4] == '0':
    dur = 100
    trans_time = 0.8
elif case[4] == '1':
    dur = 400
    trans_time = 0.95
    
maxC = [5., -5., 0.18, 0.]

n_dur = int(np.around(dur/aln.params.dt + 1.,1))
max_cntrl, min_cntrl = setmaxmincontrol(maxC[0], maxC[1], maxC[2], maxC[3])

In [9]:
init_file = 'control_init_' + case + '.pickle'
final_file = 'control_0_' + case + '.pickle'
final_file_1 = 'control_1_' + case + '.pickle'

In [10]:
if os.path.isfile(init_file) :
    print("file found")
    
    with open(init_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_init = load_array[0]
    bestState_init = load_array[1]
    cost_init = load_array[2]
    runtime_init = load_array[3]
    grad_init = load_array[4]
    phi_init = load_array[5]
    costnode_init = load_array[6]
    weights_init = load_array[7]

file found


In [11]:
i_stepsize = 10
i_range = range(0, len(exc),i_stepsize)
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = 3000.
    
    control0 = aln.getZeroControl()
    control0 = functions.step_control(aln, maxI_ = max_I[0])

    aln.run(control=control0)
    
    target_rates = np.zeros((2))
    target_rates[0] = aln.rates_exc[0,-1] 
    target_rates[1] = aln.rates_inh[0,-1]

    control0 = functions.step_control(aln, maxI_ = max_I[1])
    aln.run(control=control0)

    init_state_vars = np.zeros(( len(state_vars) ))
    for j in range(len(state_vars)):
        if aln.state[state_vars[j]].size == 1:
            init_state_vars[j] = aln.state[state_vars[j]][0]
        else:
            init_state_vars[j] = aln.state[state_vars[j]][0,-1]

    initVars[i] = init_state_vars
    
    aln.params.duration = dur

    target[i] = aln.getZeroTarget()
    target[i][:,0,:] = target_rates[0]
    target[i][:,1,:] = target_rates[1]

-------  0 0.4000000000000001 0.3500000000000001
-------  10 0.4250000000000001 0.42500000000000016
-------  20 0.4500000000000001 0.4750000000000002
-------  30 0.4250000000000001 0.5250000000000002
-------  40 0.5250000000000001 0.5500000000000003
-------  50 0.47500000000000014 0.6000000000000003
-------  60 0.5500000000000003 0.6250000000000003
-------  70 0.4500000000000001 0.6750000000000004
-------  80 0.5250000000000001 0.7000000000000004
-------  90 0.6000000000000003 0.7250000000000004
-------  100 0.4500000000000001 0.7750000000000005
-------  110 0.5000000000000002 0.8000000000000005
-------  120 0.5500000000000003 0.8250000000000005
-------  130 0.6000000000000003 0.8500000000000005
-------  140 0.4500000000000001 0.9000000000000006


In [12]:
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
    cost.setParams(1.0, 0.0, 0.0)

##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 0

    bestControl_init_, bestState_init_, cost_init_, runtime_init_, grad_init_, phi_init_, costnode_init_ = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    cost_uncontrolled[i] = cost_init_[0]

-------  0 0.4000000000000001 0.3500000000000001
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5902.406479238383
Gradient descend method:  None
RUN  0 , total integrated cost =  5902.406479238383
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  10 0.4250000000000001 0.42500000000000016
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9111.456490210901
Gradient descend method:  None
RUN  0 , total integrated cost =  9111.456490210901
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  20 0.4500000000000001 0.4750000000000002
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  12738.116450271265
Gradient descend method:  None
RUN  0 , total integrated cost =  12738.116450271265
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  30 0.4250000000000001 0.52500000000000

In [13]:
i_range_ = []

for i in i_range:
    if type(bestControl_init[i]) == type(None):
        i_range_.append(i)

i_range = np.array(i_range_)
        
print(i_range)

[ 10  20  30  40  50  60  80  90 100 110 120 130]


In [14]:
factor_iteration = 20.

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

##### initial guess
    weight_ = 10
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    #if prev_i != -1:
    #    control0 = bestControl_init[prev_i][:,:,n_pre-1:-n_post+1]
    
    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(100 * factor_iteration)
    
    weights_init[i] = cost.getParams()

    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_init[i][-j] == 0.:
        j += 1
    
    weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(500 * factor_iteration)

    weights_init[i] = cost.getParams()
    
    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    prev_i = i

-------  10 0.4250000000000001 0.42500000000000016
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9111.456490210901
Gradient descend method:  None
RUN  1 , total integrated cost =  777.7450427577537
RUN  2 , total integrated cost =  175.58970906556172
RUN  3 , total integrated cost =  136.12960985785796
RUN  4 , total integrated cost =  109.92686202528112
RUN  5 , total integrated cost =  93.03024402986364
RUN  6 , total integrated cost =  73.7954048739019
RUN  7 , total integrated cost =  59.83856346333337
RUN  8 , total integrated cost =  38.92873802793846
RUN  9 , total integrated cost =  30.018210026469344
RUN  10 , total integrated cost =  29.526698328225084
RUN  11 , total integrated cost =  29.277675215737382
RUN  12 , total integrated cost =  29.228544083791878
RUN  13 , total integrated cost =  29.222762239323636
RUN  14 , total integrated cost =  29.212863583884467
RUN  15 , total integrated cost =  29.207679217241502
RU

ERROR:root:Problem in initial value trasfer


RUN  30 , total integrated cost =  8939.992173473967
Control only changes marginally.
RUN  31 , total integrated cost =  8939.992173473967
Improved over  31  iterations in  2.2888314705342054  seconds by  1.6669889539856086  percent.
Problem in initial value trasfer:  Vmean_exc -56.645916564265896 -56.64592653944203
-------  20 0.4500000000000001 0.4750000000000002
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  12738.116450271265
Gradient descend method:  None
RUN  1 , total integrated cost =  1800.0786222783465
RUN  2 , total integrated cost =  745.0360522731675


ERROR:root:Problem in initial value trasfer


RUN  3 , total integrated cost =  745.0360522731675
Control only changes marginally.
RUN  3 , total integrated cost =  745.0360522731675
Improved over  3  iterations in  0.7293100524693727  seconds by  94.15112858182968  percent.
Problem in initial value trasfer:  Vmean_exc -56.66644541028723 -56.666516474381254
weight =  170.97315507627053
set cost params:  1.0 0.0 170.97315507627053
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  7259.863045519684
Gradient descend method:  None


...

RUN  2 , total integrated cost =  39268.69637842689
RUN  3 , total integrated cost =  39268.6961372932
RUN  4 , total integrated cost =  39268.6961169905


ERROR:root:Problem in initial value trasfer


RUN  5 , total integrated cost =  39268.69611699038
RUN  6 , total integrated cost =  39268.69611699038
Control only changes marginally.
RUN  6 , total integrated cost =  39268.69611699038
Improved over  6  iterations in  0.6183436587452888  seconds by  0.15875401641267217  percent.
Problem in initial value trasfer:  Vmean_exc -56.699650128629784 -56.699650117626796
-------  100 0.4500000000000001 0.7750000000000005
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  10559.709248318897
Gradient descend method:  None
RUN  1 , total integrated cost =  10559.709248318897
Control only changes marginally.
RUN  1 , total integrated cost =  10559.709248318897
Improved over  1  iterations in  0.13021396100521088  seconds by  0.0  percent.
weight =  10.0
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  10559.709248318897
Gradient descend method:  None
RUN  1 , total integrated cost =  10559.

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  26 , total integrated cost =  36921.90326228198
Improved over  26  iterations in  2.026304753497243  seconds by  3.9822513177140877  percent.
Problem in initial value trasfer:  Vmean_exc -56.70018887324246 -56.700188777224405


In [15]:
#plot initial guesses
"""
for i in i_range:
        
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i]],
        [costnode_init[i]], [weights_init[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

'\nfor i in i_range:\n        \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i]],\n        [costnode_init[i]], [weights_init[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n'

In [16]:
found_solution = []
no_solution = []
last_update = -1
factor_iteration = 20.
already_tried = [ [] for _ in range(len(exc)) ]
i_range = range(0, len(exc),i_stepsize)
i_range_0 = []
i_range_1 = []

print(already_tried, len(already_tried))

for k in range(len(i_range)**2):
    print('------------------------------------------------------------')
    print('--------------------', k)
    print('------------------------------------------------------------')
        
    print("found solution: ", found_solution)
    print("no solution: ", no_solution)
    
    if len(i_range) == len(found_solution) + len(no_solution):
        print("found solution for all parameters")
        break
    
    #if last_update != k-1:
    #    print("no improvement from previous step")
    #    break

    for i in i_range:
        print("------- ", i, exc[i], inh[i])

        if np.abs(bestState_init[i][0,0,-1] - target[i][0,0,-1]) < 0.5 * np.abs(
            bestState_init[i][0,0,-1] - bestState_init[i][0,0,0]):
            if i not in found_solution:
                print("found solution for ", i)
                found_solution.append(i)
                #last_update = k
            if i in no_solution:
                no_solution.pop(no_solution.index(i))
            continue

        #if i not in no_solution:
        #    print("no solution for ", i)
        #    no_solution.append(i)
        
        if i not in i_range_0:
            i_range_0.append(i)
            
        if i not in i_range_1:
            i_range_1.append(i)

        closest_ = getclosest(i, found_solution, exc, inh, already_tried[i])
        print("closest index ", closest_)

        weight_ = 10
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
            
        if i != 0 and closest_ != -1:
            control0 = bestControl_init[closest_][:,:,n_pre-1:-n_post+1]
            if closest_ not in already_tried[i]:
                already_tried[i].append(closest_)
                        
        if closest_ == -1:
            print("all options tried already")
            if i not in no_solution:
                no_solution.append(i)
                continue

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(100 * factor_iteration)

        weights_init[i] = cost.getParams()
        
        print("precision vars = ", prec_vars)

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

        j = 1
        while cost_init[i][-j] == 0.:
            j += 1

        weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(500 * factor_iteration)

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

[[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []] 147
------------------------------------------------------------
-------------------- 0
------------------------------------------------------------
found solution:  []
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
found solution for  0
-------  10 0.4250000000000001 0.42500000000000016
found solution for  10
-------  20 0.4500000000000001 0.4750000000000002
found solution for  20
-------

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  57 , total integrated cost =  740.9819984282835
Improved over  57  iterations in  14.225125320255756  seconds by  0.789018363106436  percent.
Problem in initial value trasfer:  Vmean_exc -56.6353489015496 -56.6353927372917
weight =  107.67221334268174
set cost params:  1.0 0.0 107.67221334268174
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  4663.530884857189
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  4663.530884857189
Control only changes marginally.
RUN  1 , total integrated cost =  4663.530884857189
Improved over  1  iterations in  0.6388728078454733  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.6353489015496 -56.6353927372917
-------  40 0.5250000000000001 0.5500000000000003
found solution for  40
-------  50 0.47500000000000014 0.6000000000000003
[0, 10, 20, 40] []
closest index  40
set cost params:  1.0 0.0 10.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  15975.969154530307
Gradient descend method:  None
RUN  1 , total integrated cost =  15944.309941314703
RUN  2 , total integrated cost =  73.64807109017254
RUN  3 , total integrated cost =  65.86015036406513
RUN  4 , total integrated cost =  65.56374754248097
RUN  5 , total integrated cost =  65.52119597734278
RUN  6 , total integrated cost =  65.47576859883667
RUN  7 , total integrated cost =  65.37307132013296
RU

...

ERROR:root:Problem in initial value trasfer


RUN  10000 , total integrated cost =  10554.102570479856
RUN  10000 , total integrated cost =  10554.102570479856
Improved over  10000  iterations in  673.2201817091554  seconds by  0.053045933718337324  percent.
Problem in initial value trasfer:  Vmean_exc -56.65536838389688 -56.6553684879339
-------  110 0.5000000000000002 0.8000000000000005
[0, 10, 20, 40, 60, 70, 90] []
closest index  90
set cost params:  1.0 0.0 10.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  19248.200429938483
Gradient descend method:  None
RUN  1 , total integrated cost =  19226.325434930266
RUN  2 , total integrated cost =  19226.105708425483
RUN  3 , total integrated cost =  19226.098557370067
RUN  4 , total integrated cost =  19226.098318951455
RUN  5 , total integrated cost =  19226.098318206397
RUN  6 , total integrated cost =  19226.098318201668
RUN  7 , total integrated cost =  19226.098318201533
RUN  8 , total integrated cost =  19226.098318201533
Contro

ERROR:root:Problem in initial value trasfer


RUN  10000 , total integrated cost =  422.0933137223066
RUN  10000 , total integrated cost =  422.0933137223066
Improved over  10000  iterations in  632.0128553882241  seconds by  98.523794469664  percent.
Problem in initial value trasfer:  Vmean_exc -56.70380842286756 -56.703828859661165
-------  130 0.6000000000000003 0.8500000000000005
found solution for  130
-------  140 0.4500000000000001 0.9000000000000006
found solution for  140
------------------------------------------------------------
-------------------- 1
------------------------------------------------------------
found solution:  [0, 10, 20, 40, 60, 70, 90, 130, 140]
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
-------  10 0.4250000000000001 0.42500000000000016
-------  20 0.4500000000000001 0.4750000000000002
-------  30 0.4250000000000001 0.5250000000000002
found solution for  30
-------  40 0.5250000000000001 0.5500000000000003
-------  50 0.47500000000000014 0.6000000000000003
found solution for 

RUN  6000 , total integrated cost =  19226.070228857392
RUN  7000 , total integrated cost =  19226.066410411833
RUN  8000 , total integrated cost =  19226.062590266483
RUN  9000 , total integrated cost =  19226.05877881188
RUN  10000 , total integrated cost =  19226.054965983127
RUN  10000 , total integrated cost =  19226.054965983127
Improved over  10000  iterations in  680.4585597682744  seconds by  0.00022112847165089988  percent.
-------  120 0.5500000000000003 0.8250000000000005
found solution for  120
-------  130 0.6000000000000003 0.8500000000000005
-------  140 0.4500000000000001 0.9000000000000006
------------------------------------------------------------
-------------------- 2
------------------------------------------------------------
found solution:  [0, 10, 20, 40, 60, 70, 90, 130, 140, 30, 50, 80, 100, 120]
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
-------  10 0.4250000000000001 0.42500000000000016
-------  20 0.4500000000000001 0.4750000000000

In [ ]:
factor_iteration = 20
full_converge = False
conv_init = [[False]*2] * len(exc)

for i in range(len(conv_init)):
    if i not in i_range:
        conv_init[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    if counter > 100:
        break
        
    print(conv_init[::i_stepsize])
    full_converge = True
    
    for conv in conv_init[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range:        

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_init[i][-j] == 0.:
            j += 1
            
        if j == cost_init[i].shape[0]-1:
            print("converged for ", i)
            if conv_init[i][0]:
                conv_init[i] = [True, True]
            else:
                conv_init[i] = [True, False]
            continue
    
        print("no convergence")
            
        weight_ = (factor_we * weights_init[i][1] * cost_uncontrolled[i] / cost_init[i][-j]
                   + factor_ws * weights_init[i][2] * cost_uncontrolled[i] / cost_init[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        counter += 1

[[False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False]]
-------  0 0.4000000000000001 0.3500000000000001
converged for  0
-------  10 0.4250000000000001 0.42500000000000016
no convergence
weight =  3740.257866034431
set cost params:  1.0 0.0 3740.257866034431
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9108.584186701877
Gradient descend method:  None
RUN  1 , total integrated cost =  9108.581413875392
RUN  2 , total integrated cost =  9108.581412350222
RUN  3 , total integrated cost =  9108.58141235021


ERROR:root:Problem in initial value trasfer


RUN  4 , total integrated cost =  9108.58141235021
Control only changes marginally.
RUN  4 , total integrated cost =  9108.58141235021
Improved over  4  iterations in  0.4624953046441078  seconds by  3.045864879425153e-05  percent.
Problem in initial value trasfer:  Vmean_exc -56.64591136116857 -56.64592142315793
-------  20 0.4500000000000001 0.4750000000000002
converged for  20
-------  30 0.4250000000000001 0.5250000000000002
converged for  30
-------  40 0.5250000000000001 0.5500000000000003
no convergence
weight =  7715.908117723695
set cost params:  1.0 0.0 7715.908117723695
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  25527.317472055547
Gradient descend method:  None
RUN  1 , total integrated cost =  25527.315962707686
RUN  2 , total integrated cost =  25527.315935812
RUN  3 , total integrated cost =  25527.315935811974


ERROR:root:Problem in initial value trasfer


RUN  4 , total integrated cost =  25527.315935811966
RUN  5 , total integrated cost =  25527.315935811966
Control only changes marginally.
RUN  5 , total integrated cost =  25527.315935811966
Improved over  5  iterations in  0.5581361893564463  seconds by  6.01803766642206e-06  percent.
Problem in initial value trasfer:  Vmean_exc -56.702871120961845 -56.70287121811925
-------  50 0.47500000000000014 0.6000000000000003
no convergence
weight =  2513.6467726957158
set cost params:  1.0 0.0 2513.6467726957158
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  15936.465817079032
Gradient descend method:  None
RUN  1 , total integrated cost =  15936.465390482981
RUN  2 , total integrated cost =  15936.46539048298


...

RUN  2 , total integrated cost =  24409.82570344622
RUN  3 , total integrated cost =  24409.82570282061
RUN  4 , total integrated cost =  24409.825697849512
RUN  5 , total integrated cost =  24409.82568491862
RUN  6 , total integrated cost =  24409.825683520543
RUN  7 , total integrated cost =  24409.82568266393
RUN  8 , total integrated cost =  24409.825663214175
RUN  9 , total integrated cost =  24409.825650984465
RUN  10 , total integrated cost =  24409.825650358314
RUN  11 , total integrated cost =  24409.825645404246
RUN  12 , total integrated cost =  24409.825632460357
RUN  13 , total integrated cost =  24409.825631057785
RUN  14 , total integrated cost =  24409.825630203988
RUN  15 , total integrated cost =  24409.825610791046
RUN  16 , total integrated cost =  24409.82559852357
RUN  17 , total integrated cost =  24409.825597898358
RUN  18 , total integrated cost =  24409.82559291323
RUN  19 , total integrated cost =  24409.825579993172
RUN  20 , total integrated cost =  24409.8

RUN  6000 , total integrated cost =  24409.780888569003
RUN  7000 , total integrated cost =  24409.77342418987
RUN  8000 , total integrated cost =  24409.764766808443
RUN  9000 , total integrated cost =  24395.45513521941
RUN  10000 , total integrated cost =  24395.448425827308
RUN  10000 , total integrated cost =  24395.448425827308
Improved over  10000  iterations in  596.9668490085751  seconds by  0.05889976863217328  percent.
-------  90 0.6000000000000003 0.7250000000000004
converged for  90
-------  100 0.4500000000000001 0.7750000000000005
converged for  100
-------  110 0.5000000000000002 0.8000000000000005
converged for  110
-------  120 0.5500000000000003 0.8250000000000005
converged for  120
-------  130 0.6000000000000003 0.8500000000000005
converged for  130
-------  140 0.4500000000000001 0.9000000000000006
converged for  140
[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [False, False], [True, True], [True

RUN  70 , total integrated cost =  24409.82984681003
RUN  80 , total integrated cost =  24409.829831118765
RUN  90 , total integrated cost =  24409.82982311036
RUN  100 , total integrated cost =  24409.829814092802
RUN  110 , total integrated cost =  24409.82979840138
RUN  120 , total integrated cost =  24409.829789976073
RUN  130 , total integrated cost =  24409.829776556508
RUN  140 , total integrated cost =  24409.829766995128
RUN  150 , total integrated cost =  24409.82975898661
RUN  160 , total integrated cost =  24409.829750978493
RUN  170 , total integrated cost =  24409.829741375113
RUN  180 , total integrated cost =  24409.82972568652
RUN  190 , total integrated cost =  24409.82971666285
RUN  200 , total integrated cost =  24409.82970097019
RUN  300 , total integrated cost =  24409.829606447085
RUN  400 , total integrated cost =  24409.829505130707
RUN  500 , total integrated cost =  24409.82939270605
RUN  600 , total integrated cost =  24409.82926198152
RUN  700 , total integ

RUN  2 , total integrated cost =  24409.829931049997
RUN  3 , total integrated cost =  24409.829931046617
RUN  4 , total integrated cost =  24409.829931042794
RUN  5 , total integrated cost =  24409.82993103754
RUN  6 , total integrated cost =  24409.829931031785
RUN  7 , total integrated cost =  24409.829931019172
RUN  8 , total integrated cost =  24409.829930978864
RUN  9 , total integrated cost =  24409.829929294872
RUN  10 , total integrated cost =  24409.829921723973
RUN  11 , total integrated cost =  24409.829921274522
RUN  12 , total integrated cost =  24409.8299212373
RUN  13 , total integrated cost =  24409.829921232005
RUN  14 , total integrated cost =  24409.829921227796
RUN  15 , total integrated cost =  24409.829921223507
RUN  16 , total integrated cost =  24409.82992121781
RUN  17 , total integrated cost =  24409.829921207107
RUN  18 , total integrated cost =  24409.82992118602
RUN  19 , total integrated cost =  24409.829920788026
RUN  20 , total integrated cost =  24409.

RUN  6000 , total integrated cost =  24409.821549501947
RUN  7000 , total integrated cost =  24409.820214605214
RUN  8000 , total integrated cost =  24409.818846509606
RUN  9000 , total integrated cost =  24409.81751714175
RUN  10000 , total integrated cost =  24409.816116412203
RUN  10000 , total integrated cost =  24409.816116412203
Improved over  10000  iterations in  573.9240399450064  seconds by  5.6611469133827086e-05  percent.
-------  90 0.6000000000000003 0.7250000000000004
converged for  90
-------  100 0.4500000000000001 0.7750000000000005
converged for  100
-------  110 0.5000000000000002 0.8000000000000005
converged for  110
-------  120 0.5500000000000003 0.8250000000000005
converged for  120
-------  130 0.6000000000000003 0.8500000000000005
converged for  130
-------  140 0.4500000000000001 0.9000000000000006
converged for  140
[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [False, False], [True, True], [

In [ ]:
print(conv_init[::i_stepsize])

with open(init_file,'wb') as f:
    pickle.dump([bestControl_init, bestState_init, cost_init, runtime_init, grad_init, phi_init,
                 costnode_init, weights_init], f)

In [ ]:
if os.path.isfile(final_file) :
    print("file found")
    
    with open(final_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_0 = load_array[0]
    bestState_0 = load_array[1]
    cost_0 = load_array[2]
    runtime_0 = load_array[3]
    grad_0 = load_array[4]
    phi_0 = load_array[5]
    costnode_0 = load_array[6]
    weights_0 = load_array[7]

In [ ]:
i_range_0 = range(0, len(exc),i_stepsize)
i_range_ = []

for i in i_range_0:
    if type(bestControl_0[i]) == type(None):
        i_range_.append(i)

i_range_0 = np.array(i_range_)
        
print(i_range_0)

In [ ]:
factor_iteration = 20
    
for i in i_range_0:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

# exc and inh control current 

    setinit(initVars[i], aln)
    aln.params.duration = dur
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1].copy()
    #control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]
 
    cost.setParams(1.0, 0.0, 10.0)

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_ = 100
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = "HS"
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_ = 100 * cost_uncontrolled[i] / cost_0[i][-j] - 1
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

In [ ]:
"""
for i in i_range_0:
    
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],
        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

In [ ]:
factor_iteration = 20
full_converge = False
conv_0 = [[False]*2] * len(exc)

for i in range(len(conv_0)):
    if i not in i_range_0:
        conv_0[i] = [True, True]

counter = 0

while full_converge == False:
    print('---------------', counter)
    
    if counter > 100:
        break
    
    print(conv_0[::i_stepsize])
    full_converge = True
    
    for conv in conv_0[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_0:

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.

        j = 1
        while cost_0[i][-j] == 0.:
            j += 1
            
        if j == cost_0[i].shape[0]-1:
            print("converged for ", i)
            if conv_0[i][0]:
                conv_0[i] = [True, True]
            else:
                conv_0[i] = [True, False]
            continue
    
        print("no convergence")

        weight_ = (factor_we * weights_0[i][1] * cost_uncontrolled[i] / cost_0[i][-j]
                       + factor_ws * weights_0[i][2] * cost_uncontrolled[i] / cost_0[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = 500 * factor_iteration

        weights_0[i] = cost.getParams()

        bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
    counter += 1
    

In [ ]:
print(conv_0[::i_stepsize])

with open(final_file,'wb') as f:
    pickle.dump([bestControl_0, bestState_0, cost_0, runtime_0, grad_0, phi_0,
                 costnode_0, weights_0], f)

In [ ]:
bestControl_1 = [None] * len(exc)
bestState_1 = [None] * len(exc)
cost_1 = [None] * len(exc)
runtime_1 = [None] * len(exc)
grad_1 = [None] * len(exc)
phi_1 = [None] * len(exc)
costnode_1 = [None] * len(exc)
weights_1 = [None] * len(exc)

conv_1 = [[False]*2] * len(exc)

In [ ]:
if os.path.isfile(final_file_1) :
    print("file found")
    
    with open(final_file_1,'rb') as f:
        load_array = pickle.load(f)

    bestControl_1 = load_array[0]
    bestState_1 = load_array[1]
    cost_1 = load_array[2]
    runtime_1 = load_array[3]
    grad_1 = load_array[4]
    phi_1 = load_array[5]
    costnode_1 = load_array[6]
    weights_1 = load_array[7]

In [ ]:
i_range_1 = range(0, len(exc),i_stepsize)
i_range_ = []

for i in i_range_1:
    if type(bestControl_1[i]) == type(None):
        i_range_.append(i)

i_range_1 = np.array(i_range_)  

print(i_range_1)

In [ ]:
factor_iteration = 20

for i in i_range_1:        

    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int( 500 * factor_iteration )

    weights_1[i] = cost.getParams()

    bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

In [ ]:
factor_iteration = 20
full_converge = False

for i in range(len(conv_1)):
    if i not in i_range_1:
        conv_1[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    print('---------------', counter)
    
    print(conv_1[::i_stepsize])
    full_converge = True
    
    for conv in conv_1[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_1:        

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_1[i][-j] == 0.:
            j += 1
            
        if j == cost_1[i].shape[0]-1:
            print("converged for ", i)
            if conv_1[i][0]:
                conv_1[i] = [True, True]
            else:
                conv_1[i] = [True, False]
            continue
    
        print("no convergence")
        
        cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_1[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_1[i] = cost.getParams()

        bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        counter += 1

In [ ]:
print(conv_1[::i_stepsize])

with open(final_file_1,'wb') as f:
    pickle.dump([bestControl_1, bestState_1, cost_1, runtime_1, grad_1, phi_1,
                 costnode_1, weights_1], f)